In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, LSTM, TimeDistributed
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from math import sqrt
import matplotlib.pyplot as plt

AttributeError: type object 'numpy.ndarray' has no attribute 'tostring'

In [ ]:
np.random.seed(7)
tf.random.set_seed(7)

In [ ]:
csv_path = "train.csv"
train = pd.read_csv(csv_path, parse_dates=["date"])
train = train.query("store == 1 and item == 1").copy()
train = train.sort_values("date")
print(train.head())
print(len(train))

        date  store  item  sales
0 2013-01-01      1     1     13
1 2013-01-02      1     1     11
2 2013-01-03      1     1     14
3 2013-01-04      1     1     13
4 2013-01-05      1     1     10
1826


In [ ]:
train

,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10
...,...,...,...,...
1821,2017-12-27,1,1,14
1822,2017-12-28,1,1,19
1823,2017-12-29,1,1,15
1824,2017-12-30,1,1,27


In [ ]:
def make_supervised_windows(series: np.ndarray, window: int, lag: int):
  X, y = [], []
  for i in range(window, len(series)-lag):
    X.append(series[i-window:i+1])
    y.append(series[i+lag])
  return np.array(X), np.array(y)
window = 29
lag = 1

X, y = make_supervised_windows(train["sales"].values, window, lag)
print("X",X.shape,"y", y.shape)

X (1796, 30) y (1796,)


In [ ]:
cut = int(len(X) * 0.8)
X_train, y_train = X[:cut], y[:cut]
X_test, y_test = X[cut:], y[cut:]

scaler_x = StandardScaler().fit(X_train)
scaler_y = StandardScaler().fit(y_train.reshape(-1, 1))

X_train_s = scaler_x.transform(X_train)
X_test_s = scaler_x.transform(X_test)

y_train_s = scaler_y.transform(y_train.reshape(-1,1)).ravel()
y_test_s = scaler_y.transform(y_test.reshape(-1,1)).ravel()

print("X_train_s",X_train_s, "y_train_s", y_train_s.shape)

X_train_s [[-0.98253317 -1.2868465  -0.83220313 ... -1.3187656  -2.08691314
  -1.62921404]
 [-1.28605422 -0.83143413 -0.98407042 ... -2.08733192 -1.62584223
  -1.01358313]
 [-0.83077264 -0.98323826 -1.43967229 ... -1.62619213 -1.011081
  -1.32139859]
 ...
 [ 0.68683264 -0.98323826 -1.43967229 ... -0.85762581 -0.08893917
  -0.70576768]
 [-0.98253317 -1.43865062 -0.37660127 ... -0.0890595  -0.70370039
  -1.47530632]
 [-1.43781475 -0.37602177 -0.68033584 ... -0.70391255 -1.47215192
  -0.55185995]] y_train_s (1436,)


In [ ]:
x_train_3d = X_train_s.reshape((len(X_train_s), window+1, 1))
x_test_3d = X_test_s.reshape((len(X_test_s), window+1, 1))

In [ ]:
print("Train 3d:",x_train_3d)
print("Test 3d:",x_test_3d)

Train 3d: [[[-0.98253317]
  [-1.2868465 ]
  [-0.83220313]
  ...
  [-1.3187656 ]
  [-2.08691314]
  [-1.62921404]]

 [[-1.28605422]
  [-0.83143413]
  [-0.98407042]
  ...
  [-2.08733192]
  [-1.62584223]
  [-1.01358313]]

 [[-0.83077264]
  [-0.98323826]
  [-1.43967229]
  ...
  [-1.62619213]
  [-1.011081  ]
  [-1.32139859]]

 ...

 [[ 0.68683264]
  [-0.98323826]
  [-1.43967229]
  ...
  [-0.85762581]
  [-0.08893917]
  [-0.70576768]]

 [[-0.98253317]
  [-1.43865062]
  [-0.37660127]
  ...
  [-0.0890595 ]
  [-0.70370039]
  [-1.47530632]]

 [[-1.43781475]
  [-0.37602177]
  [-0.68033584]
  ...
  [-0.70391255]
  [-1.47215192]
  [-0.55185995]]]
Test 3d: [[[-0.37549106]
  [-0.67963001]
  [-1.89527416]
  ...
  [-1.47247886]
  [-0.55001008]
  [-0.85967541]]

 [[-0.67901211]
  [-1.89406298]
  [-1.13593771]
  ...
  [-0.55019928]
  [-0.8573907 ]
  [ 0.67940187]]

 [[-1.89309633]
  [-1.13504238]
  [ 0.0790006 ]
  ...
  [-0.85762581]
  [ 0.67951236]
  [-0.85967541]]

 ...

 [[ 1.29387475]
  [-0.67963001]
 